# Seascape Data Explorer

Read one immutable product from a completed canonical Seascape release. This notebook does not download sources, build candidates, publish releases, or require OrcaCast.

## 1. Choose a workspace

Set the workspace containing a completed canonical release. Product and resolution selection is exact; the resolver never falls back to another scale.

In [ ]:
from pathlib import Path

import pandas as pd
from IPython.display import display

from seascape.products import list_products, list_resolutions, resolve_product

WORKSPACE = Path.cwd().resolve()
PRODUCT = "bathymetry"
RESOLUTION = 6

## 2. Inspect release identity and available products

In [ ]:
products = list_products(workspace=WORKSPACE)
product_table = pd.DataFrame(
    [
        {"product": product, "resolutions": list_resolutions(product, workspace=WORKSPACE)}
        for product in products
    ]
)
display(product_table)

## 3. Resolve one immutable artifact

Resolution verifies the completed release, its governed metadata, family manifests, and the selected artifact checksum before returning its identity.

In [ ]:
artifact = resolve_product(
    workspace=WORKSPACE,
    product=PRODUCT,
    resolution=RESOLUTION,
)
identity = pd.Series(
    {
        "release_id": artifact.release_id,
        "product_id": artifact.product_id,
        "dataset_id": artifact.dataset_id,
        "resolution": artifact.resolution,
        "schema_version": artifact.schema_version,
        "grain": artifact.grain,
        "path": artifact.path,
        "checksum_algorithm": artifact.checksum_algorithm,
        "checksum": artifact.checksum,
        "producer": artifact.producer,
        "manifest_path": artifact.manifest_path,
    },
    name="artifact identity",
)
display(identity.to_frame("value"))

## 4. Inspect provenance, coverage, and rights

In [ ]:
display(pd.Series(dict(artifact.producer_code_identity), name="producer code").to_frame("value"))
display(pd.Series(dict(artifact.spatial_support), name="spatial support").to_frame("value"))
display(pd.Series(dict(artifact.coverage), name="coverage").to_frame("value"))
display(pd.DataFrame([dict(item) for item in artifact.source_vintage]))
display(pd.Series(dict(artifact.rights), name="rights and attribution").to_frame("value"))

## 5. Inspect schema, missingness, and QC

The selected artifact is read only after identity verification. Missing values remain missing; the explorer does not coerce unknown, unavailable, unsurveyed, disconnected, or not-applicable states to zero.

In [ ]:
frame = pd.read_parquet(artifact.path)
schema = pd.DataFrame(
    {
        "column": frame.columns,
        "dtype": [str(frame[column].dtype) for column in frame.columns],
        "non_null": [int(frame[column].notna().sum()) for column in frame.columns],
        "null": [int(frame[column].isna().sum()) for column in frame.columns],
        "null_fraction": [float(frame[column].isna().mean()) for column in frame.columns],
    }
)
display(schema)

qc_columns = [
    column
    for column in frame.columns
    if any(token in column for token in ("QC", "STATUS", "AVAILABLE", "UNSURVEYED"))
]
for column in qc_columns:
    display(frame[column].value_counts(dropna=False).head(25).rename_axis(column).to_frame("rows"))

## 6. Summarize values

In [ ]:
numeric = frame.select_dtypes(include="number")
if numeric.empty:
    print("The selected product has no numeric columns.")
else:
    display(numeric.describe(percentiles=[0.05, 0.25, 0.5, 0.75, 0.95]).T)

display(frame.head(10))

## 7. Optional spatial inspection

Use the toolkit's family-specific `seascape inspect` command when a product has an appropriate spatial renderer. Those diagnostics are intentionally separate from this read-only tabular explorer and do not replace release validation.